In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CARPETA = "/content/drive/MyDrive/fresas"
R = "/content/recortes"
!unzip -q -o "$CARPETA/recortes.zip" -d /content
!cp "$CARPETA/etiquetas_propias.csv" /content/

In [ ]:
import random
import numpy as np
import pandas as pd

rec = pd.read_csv(f"{R}/recortes.csv")
eti = pd.read_csv("/content/etiquetas_propias.csv")
pub = rec[rec.fuente != "propias"].assign(etiqueta=lambda d: d.clase)
pro = eti[eti.etiqueta.isin(["sana", "danada"])].assign(fuente="propias")
columnas = ["recorte", "fuente", "grupo", "etiqueta"]
datos = pd.concat([pub[columnas], pro[columnas]], ignore_index=True)

rng = random.Random(42)
parte = {}
for _, g in pub.groupby(["fuente", "clase"]):
    grupos = sorted(g.grupo.unique())
    rng.shuffle(grupos)
    n = max(1, round(len(grupos) * 0.15))
    for i, gr in enumerate(grupos):
        parte[gr] = "prueba" if i < n else "validacion" if i < 2 * n else "entrenamiento"
for e in pro.grupo.unique():
    parte[e] = "validacion" if e in ["escena02", "escena07", "escena11"] else "entrenamiento"
datos["parte"] = datos.grupo.map(parte)
datos["y"] = (datos.etiqueta == "danada").astype(int)

assert datos.parte.notna().all()
assert datos.groupby("grupo").parte.nunique().max() == 1
print(datos.groupby(["parte", "fuente", "etiqueta"]).size().unstack(fill_value=0))

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

disp = "cuda" if torch.cuda.is_available() else "cpu"
TAM = 224
MEDIA, DESV = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
sin_aumento = transforms.Compose([
    transforms.Resize((TAM, TAM)), transforms.ToTensor(), transforms.Normalize(MEDIA, DESV)])
con_aumento = transforms.Compose([
    transforms.RandomResizedCrop(TAM, scale=(0.7, 1), ratio=(0.8, 1.25)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(20),
    transforms.ColorJitter(0.25, 0.25, 0.15, 0.02),
    transforms.ToTensor(), transforms.Normalize(MEDIA, DESV)])


class Recortes(Dataset):
    def __init__(self, tabla, variante, trans):
        self.rutas = [f"{R}/{variante}/{r}" for r in tabla.recorte]
        self.y = tabla.y.tolist()
        self.trans = trans

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.trans(Image.open(self.rutas[i]).convert("RGB")), self.y[i]


def cargador(tabla, variante, trans, mezclar=False):
    return DataLoader(Recortes(tabla, variante, trans), batch_size=32, shuffle=mezclar, num_workers=2)


def modelo_nuevo():
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    return m.to(disp)


def predecir(m, datos_cargados):
    m.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in datos_cargados:
            ps.append(torch.softmax(m(x.to(disp)), 1)[:, 1].cpu())
            ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()


def metricas(y, p, umbral=0.5):
    pred = (p >= umbral).astype(int)
    vp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    vn = int(((pred == 0) & (y == 0)).sum())
    pr = vp / (vp + fp) if vp + fp else 0.0
    rc = vp / (vp + fn) if vp + fn else 0.0
    f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
    return {"exactitud": (vp + vn) / len(y), "precision": pr, "recall": rc, "f1": f1,
            "vp": vp, "fp": fp, "fn": fn, "vn": vn}


m_ = metricas(np.array([1, 1, 0, 0]), np.array([0.9, 0.2, 0.7, 0.1]))
assert (m_["vp"], m_["fn"], m_["fp"], m_["vn"]) == (1, 1, 1, 1) and m_["f1"] == 0.5
print("Listo en", disp)

In [ ]:
import os, time
EPOCAS = 8
os.makedirs(f"{CARPETA}/modelos", exist_ok=True)
ent, val, pru = (datos[datos.parte == p] for p in ["entrenamiento", "validacion", "prueba"])
pesos = torch.tensor(len(ent) / (2 * np.bincount(ent.y)), dtype=torch.float).to(disp)

archivo = f"{CARPETA}/modelos/resultados_validacion.csv"
resultados = pd.read_csv(archivo).to_dict("records") if os.path.exists(archivo) else []
for variante in ["caja", "negro"]:
    for aumento in [False, True]:
        nombre = f"{variante}_{'con' if aumento else 'sin'}_aumento"
        if nombre in [r["experimento"] for r in resultados]:
            print(nombre, "ya estaba en Drive")
            continue
        torch.manual_seed(0)
        c_ent = cargador(ent, variante, con_aumento if aumento else sin_aumento, mezclar=True)
        c_val = cargador(val, variante, sin_aumento)
        m = modelo_nuevo()
        opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCAS)
        perdida = nn.CrossEntropyLoss(weight=pesos)
        mejor, t0 = -1, time.time()
        for ep in range(EPOCAS):
            m.train()
            for x, y in c_ent:
                opt.zero_grad()
                perdida(m(x.to(disp)), y.to(disp)).backward()
                opt.step()
            sched.step()
            f1 = metricas(*predecir(m, c_val))["f1"]
            print(nombre, "época", ep + 1, "F1 validación", round(f1, 3))
            if f1 > mejor:
                mejor = f1
                torch.save(m.state_dict(), f"{CARPETA}/modelos/{nombre}.pt")
        resultados.append({"experimento": nombre, "variante": variante, "aumento": aumento,
                           "f1_validacion": mejor, "minutos": (time.time() - t0) / 60})
        pd.DataFrame(resultados).to_csv(archivo, index=False)
print(pd.DataFrame(resultados).round(3))

In [ ]:
import json, shutil
resultados = pd.read_csv(f"{CARPETA}/modelos/resultados_validacion.csv").to_dict("records")
filas = []
for r in resultados:
    m = modelo_nuevo()
    m.load_state_dict(torch.load(f"{CARPETA}/modelos/{r['experimento']}.pt", map_location=disp))
    for nombre_parte, tabla in [("validacion", val), ("prueba", pru)]:
        y, p = predecir(m, cargador(tabla, r["variante"], sin_aumento))
        filas.append({"experimento": r["experimento"], "parte": nombre_parte, **metricas(y, p)})
tabla_res = pd.DataFrame(filas)
tabla_res.to_csv(f"{CARPETA}/modelos/resultados.csv", index=False)
print(tabla_res.round(3).to_string(index=False))

mejor = max(resultados, key=lambda r: r["f1_validacion"])
print()
print("Mejor experimento según validación:", mejor["experimento"])
m = modelo_nuevo()
m.load_state_dict(torch.load(f"{CARPETA}/modelos/{mejor['experimento']}.pt", map_location=disp))
for fuente, tabla in pru.groupby("fuente"):
    y, p = predecir(m, cargador(tabla, mejor["variante"], sin_aumento))
    print("Prueba", fuente, {k: round(v, 3) for k, v in metricas(y, p).items()})
y, p = predecir(m, cargador(val[val.fuente == "propias"], mejor["variante"], sin_aumento))
print("Validación propias", {k: round(v, 3) for k, v in metricas(y, p).items()})

shutil.copy(f"{CARPETA}/modelos/{mejor['experimento']}.pt", f"{CARPETA}/clasificador.pt")
json.dump({"experimento": mejor["experimento"], "variante": mejor["variante"], "clases": ["sana", "danada"],
           "tamano": TAM, "media": MEDIA, "desviacion": DESV, "umbral": 0.5},
          open(f"{CARPETA}/clasificador.json", "w"), indent=1)
print("Guardado en Drive:", CARPETA + "/clasificador.pt")